In [0]:
%sql
-- ============================================================
-- ETL: gold.dim_tiempo
-- Patrón: INSERT OVERWRITE (idempotente y atómico)
-- Se recarga completa en cada corrida para cubrir el rango
-- dinámico de fechas que exista en silver.ventas.
-- ============================================================

INSERT OVERWRITE iniciacion_deportiva.gold.dim_tiempo (
    tiempo_key,
    fecha,
    dia,
    mes,
    nombre_mes,
    trimestre,
    semestre,
    anio,
    _created_at
)

WITH

-- =====================================================
-- PASO 1: Obtener rango de fechas desde Silver
-- Min y max de fechas reales de ventas
-- =====================================================
rango AS (
    SELECT
        CAST(MIN(fecha) AS DATE) AS fecha_min,
        CAST(MAX(fecha) AS DATE) AS fecha_max
    FROM iniciacion_deportiva.silver.ventas
    WHERE fecha IS NOT NULL
),

-- =====================================================
-- PASO 2: Generar secuencia de días
-- SEQUENCE genera un array, EXPLODE lo convierte en filas
-- Una fila por cada día entre fecha_min y fecha_max
-- =====================================================
dias AS (
    SELECT EXPLODE(
        SEQUENCE(fecha_min, fecha_max, INTERVAL 1 DAY)
    ) AS fecha
    FROM rango
)

-- =====================================================
-- PASO 3: Calcular atributos de cada fecha
-- =====================================================
SELECT
    CAST(DATE_FORMAT(fecha, 'yyyyMMdd') AS INT)  AS tiempo_key,
    fecha,
    DAY(fecha)                                   AS dia,
    MONTH(fecha)                                 AS mes,
    CASE MONTH(fecha)
        WHEN 1  THEN 'Enero'
        WHEN 2  THEN 'Febrero'
        WHEN 3  THEN 'Marzo'
        WHEN 4  THEN 'Abril'
        WHEN 5  THEN 'Mayo'
        WHEN 6  THEN 'Junio'
        WHEN 7  THEN 'Julio'
        WHEN 8  THEN 'Agosto'
        WHEN 9  THEN 'Septiembre'
        WHEN 10 THEN 'Octubre'
        WHEN 11 THEN 'Noviembre'
        WHEN 12 THEN 'Diciembre'
    END                                          AS nombre_mes,
    QUARTER(fecha)                               AS trimestre,
    CASE WHEN MONTH(fecha) <= 6 THEN 1 ELSE 2 END AS semestre,
    YEAR(fecha)                                  AS anio,
    CURRENT_TIMESTAMP()                          AS _created_at
FROM dias
ORDER BY fecha;